In [ ]:
# @title 1. Install & Configure
!pip install -q internetarchive rq redis pandas tqdm python-dotenv

import os
from dotenv import load_dotenv
import json
import time
from tqdm.notebook import tqdm
import pandas as pd
from internetarchive import search_items, get_item
import requests
from rq import Queue
from redis import Redis

load_dotenv()  # For .env with REDIS_URL if using external Redis

# === CONFIG ===
QUERY = 'collection:library_of_congress AND mediatype:texts AND year:1800-1900'  # ← Customize!
MAX_ITEMS = 50  # Start small for testing
OUTPUT_JSONL = 'data/raw/ia_items.jsonl'
REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')  # Change for production
QUEUE_NAME = 'ia-reconcile'

# IA polite rate limit
RATE_LIMIT = 1.0  # seconds between calls

print("✅ Setup complete. Ready for search & harvest.")

## Harvester Core (Search → Fetch Metadata + IIIF)

In [ ]:
# @title 2. Run Harvester
redis_conn = Redis.from_url(REDIS_URL)
q = Queue(QUEUE_NAME, connection=redis_conn)

items_data = []
search_results = search_items(QUERY, fields=['identifier', 'title', 'creator', 'date', 'subject'])

for item in tqdm(list(search_results.iter_as_items())[:MAX_ITEMS], desc="Harvesting IA items"):
    item_id = item['identifier']
    try:
        # 1. Full Metadata API
        meta_url = f"https://archive.org/metadata/{item_id}"
        meta = requests.get(meta_url, timeout=10).json()

        # 2. IIIF Manifest (latest 3.0)
        manifest_url = f"https://iiif.archive.org/iiif/{item_id}/manifest.json"
        manifest = requests.get(manifest_url, timeout=10).json()

        payload = {
            "item_id": item_id,
            "metadata": meta.get("metadata", {}),
            "files": meta.get("files", []),
            "iiif_manifest_url": manifest_url,
            "iiif_manifest": manifest,  # Full if small; else just URL in prod
            "harvested_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }

        # Save locally (GitHub sync)
        with open(OUTPUT_JSONL, 'a') as f:
            f.write(json.dumps(payload) + '\n')

        # 3. Enqueue to Reconciler (interoperability!)
        job = q.enqueue('reconciler.tasks.reconcile_item', payload, job_timeout='10m')
        print(f"✅ Enqueued {item_id} → Job ID: {job.id}")

        items_data.append(payload)
        time.sleep(RATE_LIMIT)

    except Exception as e:
        print(f"⚠️ Error on {item_id}: {e}")
        continue

print(f"\n🎉 Harvest complete! {len(items_data)} items saved + enqueued.")
pd.DataFrame(items_data).head()

## Monitor & Utilities

In [ ]:
# @title 3. Queue Status & Manual Re-run
print(f"Queue length: {q.count}")
# Example: Re-run a single ItemID manually
# payload = {...}  # from JSONL
# q.enqueue('reconciler.tasks.reconcile_item', payload)